In [ ]:
# ================================================================================
#
#
# ================================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.stats import binomtest

# ================================================================
# ================================================================
TASK = "triclass"
SEED = 42
BASE_DIR = "analysis/taste_baseline_cv"
TASK_DIR = f"{BASE_DIR}/{TASK}"

FP_PROPERTY_PROBA_CSV = f"{TASK_DIR}/schnet_fp_property_seed{SEED}/schnet_fp_property_oof_probabilities_{TASK}_seed{SEED}.csv"
RF_PROBA_CSV = f"{TASK_DIR}/morgan_rf_oof_probabilities_{TASK}_seed{SEED}.csv"

NUM_CLASSES = 2 if TASK == "binary" else 3


# ================================================================
# ================================================================
def metric_dict(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


def paired_bootstrap_difference(y_true, pred_a, pred_b, metric="f1_macro", B=5000, seed=42):
    y_true, pred_a, pred_b = np.asarray(y_true), np.asarray(pred_a), np.asarray(pred_b)
    rng = np.random.default_rng(seed)
    n = len(y_true)

    def score(y, p):
        if metric == "accuracy":
            return accuracy_score(y, p)
        if metric == "f1_weighted":
            return f1_score(y, p, average="weighted", zero_division=0)
        return f1_score(y, p, average="macro", zero_division=0)

    observed = score(y_true, pred_a) - score(y_true, pred_b)
    diffs = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        diffs[b] = score(y_true[idx], pred_a[idx]) - score(y_true[idx], pred_b[idx])
    lo, hi = np.quantile(diffs, [0.025, 0.975])
    p_two_sided = min(1.0, 2.0 * min(np.mean(diffs <= 0), np.mean(diffs >= 0)))
    return {"metric": metric, "difference_a_minus_b": observed, "ci95_low": lo, "ci95_high": hi,
            "p_bootstrap_two_sided": p_two_sided, "significant_0.05": bool((lo > 0) or (hi < 0))}


def exact_mcnemar(y_true, pred_a, pred_b):
    y_true = np.asarray(y_true)
    correct_a = np.asarray(pred_a) == y_true
    correct_b = np.asarray(pred_b) == y_true
    n01 = int(np.sum((~correct_a) & correct_b))
    n10 = int(np.sum(correct_a & (~correct_b)))
    discordant = n01 + n10
    p = 1.0 if discordant == 0 else binomtest(min(n01, n10), discordant, 0.5, alternative="two-sided").pvalue
    return {"n01_a_wrong_b_right": n01, "n10_a_right_b_wrong": n10, "p_exact": p}


def get_proba(df, prefix):
    cols = [f"{prefix}_class{c}" for c in range(NUM_CLASSES)]
    return df[cols].to_numpy()


def lofo_weighted_ensemble(proba_a, proba_b, y_true, folds, weight_grid=np.linspace(0, 1, 21)):
    unique_folds = np.unique(folds)
    oof_pred = np.full(len(y_true), -1, dtype=int)
    chosen_weights = {}
    for held_fold in unique_folds:
        train_mask = folds != held_fold
        test_mask = folds == held_fold
        best_w, best_f1 = None, -1
        for w in weight_grid:
            blended = w * proba_a[train_mask] + (1 - w) * proba_b[train_mask]
            pred = blended.argmax(axis=1)
            f1m = f1_score(y_true[train_mask], pred, average="macro", zero_division=0)
            if f1m > best_f1:
                best_f1, best_w = f1m, w
        chosen_weights[int(held_fold)] = best_w
        blended_test = best_w * proba_a[test_mask] + (1 - best_w) * proba_b[test_mask]
        oof_pred[test_mask] = blended_test.argmax(axis=1)
    return oof_pred, chosen_weights


def lofo_stacking(feature_matrix, y_true, folds, seed=42):
    unique_folds = np.unique(folds)
    oof_pred = np.full(len(y_true), -1, dtype=int)
    for held_fold in unique_folds:
        train_mask = folds != held_fold
        test_mask = folds == held_fold
        clf = LogisticRegression(max_iter=2000, random_state=seed)
        clf.fit(feature_matrix[train_mask], y_true[train_mask])
        oof_pred[test_mask] = clf.predict(feature_matrix[test_mask])
    return oof_pred


def main():
    fp_df = pd.read_csv(FP_PROPERTY_PROBA_CSV)
    rf_df = pd.read_csv(RF_PROBA_CSV)
    df = fp_df.merge(rf_df, on="sample_idx", suffixes=("", "_rf"))
    assert (df["true_label"] == df["true_label_rf"]).all(), "true_label differs: check sample_idx alignment"
    assert (df["fold"] == df["fold_rf"]).all(), "fold assignment differs: check that the same fold JSON was used"

    y_true = df["true_label"].to_numpy()
    folds = df["fold"].to_numpy()
    proba_fp = get_proba(df, "proba_fp_property")
    proba_rf = get_proba(df, "proba_morgan_rf")

    pred_fp = proba_fp.argmax(axis=1)
    pred_rf = proba_rf.argmax(axis=1)

    rows = [
        {"model": "SchNet (3D + Properties + FP-PCA)", **metric_dict(y_true, pred_fp)},
        {"model": "Morgan-RF", **metric_dict(y_true, pred_rf)},
    ]

    pred_ens_weighted, chosen_w = lofo_weighted_ensemble(proba_fp, proba_rf, y_true, folds)
    rows.append({"model": "Ensemble: SchNet(FP+Prop) + Morgan-RF (LOFO weighted)",
                 **metric_dict(y_true, pred_ens_weighted)})
    print("Selected weight per fold (SchNet weight; RF weight is 1-w):", chosen_w)

    pred_ens_avg = (proba_fp + proba_rf).argmax(axis=1)
    rows.append({"model": "Ensemble: SchNet(FP+Prop) + Morgan-RF (simple avg)",
                 **metric_dict(y_true, pred_ens_avg)})

    stack_features = np.concatenate([proba_fp, proba_rf], axis=1)
    pred_stack = lofo_stacking(stack_features, y_true, folds, seed=SEED)
    rows.append({"model": "Stacking (logreg on FP+Prop & RF probs, LOFO)",
                 **metric_dict(y_true, pred_stack)})

    summary = pd.DataFrame(rows)
    print("\n=== Performance by model ===")
    print(summary.round(4).to_string(index=False))

    best_row = summary.loc[summary["f1_macro"].idxmax()]
    best_name = best_row["model"]
    best_pred = {
        "Ensemble: SchNet(FP+Prop) + Morgan-RF (LOFO weighted)": pred_ens_weighted,
        "Ensemble: SchNet(FP+Prop) + Morgan-RF (simple avg)": pred_ens_avg,
        "Stacking (logreg on FP+Prop & RF probs, LOFO)": pred_stack,
        "Morgan-RF": pred_rf,
        "SchNet (3D + Properties + FP-PCA)": pred_fp,
    }[best_name]

    print(f"\n=== Best model: {best_name} (macro-F1={best_row['f1_macro']:.4f}) ===")

    print("\n--- Significance: best model vs Morgan-RF alone ---")
    for m in ["accuracy", "f1_weighted", "f1_macro"]:
        print(paired_bootstrap_difference(y_true, best_pred, pred_rf, metric=m, seed=SEED))
    print(exact_mcnemar(y_true, pred_rf, best_pred))

    print("\n--- Significance: best model vs SchNet(3D+Properties+FP-PCA) alone ---")
    for m in ["accuracy", "f1_weighted", "f1_macro"]:
        print(paired_bootstrap_difference(y_true, best_pred, pred_fp, metric=m, seed=SEED))
    print(exact_mcnemar(y_true, pred_fp, best_pred))

    out_path = FP_PROPERTY_PROBA_CSV.replace(
        "schnet_fp_property_oof_probabilities", "ensemble_fp_property_rf_summary"
    )
    summary.to_csv(out_path, index=False)
    print(f"\nSummary saved: {out_path}")


if __name__ == "__main__":
    main()


In [ ]:
# ================================================================================
#
#
# ================================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

# ================================================================
# CONFIG
# ================================================================
TASK = "triclass"
SEED = 42
BASE_DIR = "analysis/taste_baseline_cv"
TASK_DIR = f"{BASE_DIR}/{TASK}"

FP_PROPERTY_PROBA_CSV = f"{TASK_DIR}/schnet_fp_property_seed{SEED}/schnet_fp_property_oof_probabilities_{TASK}_seed{SEED}.csv"
RF_PROBA_CSV = f"{TASK_DIR}/morgan_rf_oof_probabilities_{TASK}_seed{SEED}.csv"

NUM_CLASSES = 3
CLASS_NAMES = ["Bitter", "Sweet", "Tasteless"]  # LABEL_MAP_TRI = {bitterness:0, sweetness:1, tastelessness:2}
FOCUS_CLASS = 2  # Tasteless

B_BOOTSTRAP = 5000


def get_proba(df, prefix):
    cols = [f"{prefix}_class{c}" for c in range(NUM_CLASSES)]
    return df[cols].to_numpy()


def lofo_weighted_ensemble(proba_a, proba_b, y_true, folds, weight_grid=np.linspace(0, 1, 21)):
    unique_folds = np.unique(folds)
    oof_pred = np.full(len(y_true), -1, dtype=int)
    chosen_weights = {}
    for held_fold in unique_folds:
        train_mask = folds != held_fold
        test_mask = folds == held_fold
        best_w, best_f1 = None, -1
        for w in weight_grid:
            blended = w * proba_a[train_mask] + (1 - w) * proba_b[train_mask]
            pred = blended.argmax(axis=1)
            f1m = f1_score(y_true[train_mask], pred, average="macro", zero_division=0)
            if f1m > best_f1:
                best_f1, best_w = f1m, w
        chosen_weights[int(held_fold)] = best_w
        blended_test = best_w * proba_a[test_mask] + (1 - best_w) * proba_b[test_mask]
        oof_pred[test_mask] = blended_test.argmax(axis=1)
    return oof_pred, chosen_weights


def class_f1(y_true, y_pred, class_idx):
    return f1_score(y_true, y_pred, labels=[class_idx], average=None, zero_division=0)[0]


def class_precision(y_true, y_pred, class_idx):
    return precision_score(y_true, y_pred, labels=[class_idx], average=None, zero_division=0)[0]


def class_recall(y_true, y_pred, class_idx):
    return recall_score(y_true, y_pred, labels=[class_idx], average=None, zero_division=0)[0]


def bootstrap_metric_ci(y_true, y_pred, metric_fn, B=B_BOOTSTRAP, alpha=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    n = len(y_true)
    boots = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, n)
        boots[b] = metric_fn(y_true[idx], y_pred[idx])
    point = metric_fn(y_true, y_pred)
    lo, hi = np.quantile(boots, [alpha / 2, 1 - alpha / 2])
    return point, lo, hi


def paired_bootstrap_class_metric(y_true, pred_a, pred_b, metric_fn, class_idx, B=B_BOOTSTRAP, seed=SEED):
    y_true, pred_a, pred_b = np.asarray(y_true), np.asarray(pred_a), np.asarray(pred_b)
    rng = np.random.default_rng(seed)
    n = len(y_true)
    observed = metric_fn(y_true, pred_a, class_idx) - metric_fn(y_true, pred_b, class_idx)
    diffs = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, n)
        diffs[b] = metric_fn(y_true[idx], pred_a[idx], class_idx) - metric_fn(y_true[idx], pred_b[idx], class_idx)
    lo, hi = np.quantile(diffs, [0.025, 0.975])
    p_two_sided = min(1.0, 2.0 * min(np.mean(diffs <= 0), np.mean(diffs >= 0)))
    return {"difference_a_minus_b": observed, "ci95_low": lo, "ci95_high": hi,
            "p_bootstrap_two_sided": p_two_sided, "significant_0.05": bool((lo > 0) or (hi < 0))}


def print_per_class_report(name, y_true, y_pred):
    print(f"\n--- {name}: per-class metrics (bootstrap 95% CI) ---")
    for c in range(NUM_CLASSES):
        n_c = int((y_true == c).sum())
        p_pt, p_lo, p_hi = bootstrap_metric_ci(y_true, y_pred, lambda a, b: class_precision(a, b, c))
        r_pt, r_lo, r_hi = bootstrap_metric_ci(y_true, y_pred, lambda a, b: class_recall(a, b, c))
        f_pt, f_lo, f_hi = bootstrap_metric_ci(y_true, y_pred, lambda a, b: class_f1(a, b, c))
        marker = "  <== FOCUS" if c == FOCUS_CLASS else ""
        print(f"  {CLASS_NAMES[c]:10s} (n={n_c:3d}) | "
              f"precision={p_pt:.3f} [{p_lo:.3f},{p_hi:.3f}] | "
              f"recall={r_pt:.3f} [{r_lo:.3f},{r_hi:.3f}] | "
              f"f1={f_pt:.3f} [{f_lo:.3f},{f_hi:.3f}]{marker}")

    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    print(f"  confusion matrix (rows=true, cols=pred, order={CLASS_NAMES}):")
    print(" ", cm)
    focus_row = cm[FOCUS_CLASS]
    total = focus_row.sum()
    if total > 0:
        breakdown = ", ".join(f"{CLASS_NAMES[j]}={focus_row[j]} ({100*focus_row[j]/total:.1f}%)"
                               for j in range(NUM_CLASSES))
        print(f"  Predicted-class distribution for true {CLASS_NAMES[FOCUS_CLASS]}: {breakdown}")


def main():
    fp_df = pd.read_csv(FP_PROPERTY_PROBA_CSV)
    rf_df = pd.read_csv(RF_PROBA_CSV)
    df = fp_df.merge(rf_df, on="sample_idx", suffixes=("", "_rf"))
    assert (df["true_label"] == df["true_label_rf"]).all(), "true_label mismatch: check sample_idx alignment"
    assert (df["fold"] == df["fold_rf"]).all(), "fold mismatch: check that the same fold JSON was used"

    y_true = df["true_label"].to_numpy()
    folds = df["fold"].to_numpy()
    proba_fp = get_proba(df, "proba_fp_property")
    proba_rf = get_proba(df, "proba_morgan_rf")

    pred_fp = proba_fp.argmax(axis=1)
    pred_rf = proba_rf.argmax(axis=1)
    pred_ens, chosen_w = lofo_weighted_ensemble(proba_fp, proba_rf, y_true, folds)
    print("Selected weight per fold (SchNet weight; RF weight is 1-w):", chosen_w)

    print_per_class_report("SchNet (3D + Properties + FP-PCA) alone", y_true, pred_fp)
    print_per_class_report("Morgan-RF alone", y_true, pred_rf)
    print_per_class_report("Ensemble (LOFO weighted)", y_true, pred_ens)

    print(f"\n=== {CLASS_NAMES[FOCUS_CLASS]} F1: ensemble vs Morgan-RF (paired bootstrap) ===")
    print(paired_bootstrap_class_metric(y_true, pred_ens, pred_rf, class_f1, FOCUS_CLASS))

    print(f"\n=== {CLASS_NAMES[FOCUS_CLASS]} recall: ensemble vs Morgan-RF (paired bootstrap) ===")
    print(paired_bootstrap_class_metric(y_true, pred_ens, pred_rf, class_recall, FOCUS_CLASS))

    print(f"\n=== {CLASS_NAMES[FOCUS_CLASS]} precision: ensemble vs Morgan-RF (paired bootstrap) ===")
    print(paired_bootstrap_class_metric(y_true, pred_ens, pred_rf, class_precision, FOCUS_CLASS))

    print(f"\n=== {CLASS_NAMES[FOCUS_CLASS]} F1: ensemble vs SchNet(FP+Property) alone ===")
    print(paired_bootstrap_class_metric(y_true, pred_ens, pred_fp, class_f1, FOCUS_CLASS))


if __name__ == "__main__":
    main()


In [ ]:
# ================================================================
# ================================================================
import glob, re
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support as prfs

TASK = "triclass"   # "binary" | "triclass"
BASE_DIR = "analysis/taste_baseline_cv"
TASK_DIR = f"{BASE_DIR}/{TASK}"
RF_CSV = f"{TASK_DIR}/morgan_rf_oof_probabilities_{TASK}_seed42.csv"
EXTRA_RUNS = {"42": f"{TASK_DIR}/schnet_fp_property_seed42/schnet_fp_property_oof_probabilities_{TASK}_seed42.csv"}
OUT_CSV = f"{TASK_DIR}/multiseed_summary_{TASK}.csv"

rf = pd.read_csv(RF_CSV).sort_values("sample_idx")
y = rf.true_label.values; fold = rf.fold.values
Prf = rf.filter(like="proba_").values; prf = Prf.argmax(1)
mf = lambda p: f1_score(y, p, average="macro")

runs = dict(EXTRA_RUNS)
for p in glob.glob(f"{TASK_DIR}/schnet_fp_property_trainseed*/schnet_fp_property_oof_probabilities_{TASK}_seed*.csv"):
    runs[re.search(r"trainseed(\d+)", p).group(1)] = p

rows = []
for s, path in sorted(runs.items(), key=lambda kv: int(kv[0])):
    try: sc = pd.read_csv(path).sort_values("sample_idx")
    except FileNotFoundError: print("missing", path); continue
    assert (sc.true_label.values == y).all() and (sc.fold.values == fold).all()
    Psc = sc.filter(like="proba_").values; psc = Psc.argmax(1)
    pe = np.zeros(len(y), int); ws = np.linspace(0, 1, 21); wl = []
    for k in np.unique(fold):
        tr = fold != k
        w = max(ws, key=lambda w: f1_score(y[tr], (w*Psc[tr]+(1-w)*Prf[tr]).argmax(1), average="macro"))
        wl.append(w); pe[~tr] = (w*Psc[~tr]+(1-w)*Prf[~tr]).argmax(1)
    r = {"train_seed": s, "schnet_acc": accuracy_score(y, psc), "schnet_mF1": mf(psc),
         "rf_mF1": mf(prf), "ens_acc": accuracy_score(y, pe), "ens_mF1": mf(pe),
         "ens_minus_rf_mF1": mf(pe) - mf(prf), "mean_schnet_weight": float(np.mean(wl))}
    if TASK == "triclass":
        for nm, p in [("schnet", psc), ("rf", prf), ("ens", pe)]:
            P, R, F, _ = prfs(y, p, labels=[2], zero_division=0)
            r.update({f"{nm}_tl_P": P[0], f"{nm}_tl_R": R[0], f"{nm}_tl_F1": F[0], f"{nm}_tl_TP": int(((p == 2) & (y == 2)).sum())})
        r["ens_minus_rf_tl_R"] = r["ens_tl_R"] - r["rf_tl_R"]
        r["ens_minus_rf_tl_F1"] = r["ens_tl_F1"] - r["rf_tl_F1"]
    rows.append(r)

df = pd.DataFrame(rows)
num = df.drop(columns="train_seed")
summ = pd.DataFrame({"mean": num.mean(), "sd": num.std(ddof=1)}).T
summ.insert(0, "train_seed", ["mean", "sd"])
pd.concat([df, summ]).to_csv(OUT_CSV, index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 50)
print(df.round(4).to_string(index=False)); print("\n", summ.round(4).to_string(index=False)); print("Saved:", OUT_CSV)


In [ ]:
# Sensitivity of the ensemble to a fixed SchNet blending weight (ternary task)
import glob, re
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, precision_recall_fscore_support as prfs

TASK = "triclass"
TASK_DIR = f"analysis/taste_baseline_cv/{TASK}"
RF_CSV = f"{TASK_DIR}/morgan_rf_oof_probabilities_{TASK}_seed42.csv"
FIXED_WEIGHTS = [0.10, 0.25, 0.50]

rf = pd.read_csv(RF_CSV).sort_values("sample_idx")
y = rf.true_label.values
Prf = rf.filter(like="proba_").values
prf = Prf.argmax(1)
P0, R0, F0, _ = prfs(y, prf, labels=[2], zero_division=0)

runs = {"42": f"{TASK_DIR}/schnet_fp_property_seed42/schnet_fp_property_oof_probabilities_{TASK}_seed42.csv"}
for p in glob.glob(f"{TASK_DIR}/schnet_fp_property_trainseed*/schnet_fp_property_oof_probabilities_{TASK}_seed*.csv"):
    runs[re.search(r"trainseed(\d+)", p).group(1)] = p

rows = []
for seed, path in sorted(runs.items(), key=lambda kv: int(kv[0])):
    sc = pd.read_csv(path).sort_values("sample_idx")
    Psc = sc.filter(like="proba_").values
    for w in FIXED_WEIGHTS + [1.0]:
        pe = (w * Psc + (1 - w) * Prf).argmax(1)
        P, R, F, _ = prfs(y, pe, labels=[2], zero_division=0)
        rows.append({"train_seed": int(seed), "w_schnet": w, "macro_F1": f1_score(y, pe, average="macro"),
                     "d_macro_F1": f1_score(y, pe, average="macro") - f1_score(y, prf, average="macro"),
                     "tl_recall": R[0], "d_tl_recall": R[0] - R0[0], "tl_precision": P[0], "tl_F1": F[0],
                     "tl_TP": int(((pe == 2) & (y == 2)).sum())})
fw = pd.DataFrame(rows)
summary = fw.drop(columns="train_seed").groupby("w_schnet").agg(["mean", "std"]).round(3)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(summary)
fw.to_csv(f"{TASK_DIR}/fixed_weight_sensitivity_{TASK}.csv", index=False)
